# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-04-14

### Surrogate model training script (REFACTORED)
**Goal:** Train a Graph Neural Network surrogate model on structural analysis data to predict beam forces.

**Architecture:** This notebook now uses a modular design:
- **`c21_train.py`**: Core training logic (data loading, processing, training, export)
- **`c21_surrogate_model_training.ipynb`**: Orchestration and interactive visualization

**Usage:**
1. Set environment variables or use defaults (see `c21_train.py`)
2. Run the training cell (Cell 2) → executes complete workflow
3. Run visualization cells (Cells 3-5) → generate diagnostic plots
4. Run evaluation cell (Cell 6) → export metrics

**Inputs:**
- CSV files with structural properties from Grasshopper (node, edge, global data)

**Outputs:**
- Trained surrogate model checkpoint
- Scalers for inference
- Evaluation metrics and diagnostic plots


# TRAINING EXECUTION

All training logic is encapsulated in src/c21_train.py. Run the cell below to execute the complete workflow (data loading -> processing -> training -> export). Parameters are set via environment variables (see src/c21_train.py or set via os.environ before running).


In [ ]:
"""
RUN TRAINING WORKFLOW - Single cell execution
This notebook now uses src/c21_train.py for core logic.
"""
from src.c21_train import main

# Execute training
results = main()

# Extract results for visualization cells below
model = results["model"]
device = results["device"]
epoch_history = results["epoch_history"]
train_loss_history = results["train_loss_history"]
final_val_r2 = results["final_val_r2"]
edge_target_scaler = results["scalers"]["edge_target"]
train_loader = results["train_loader"]
test_loader = results["test_loader"]
schema = results["schema"]
params = results["params"]

print("Training workflow complete. Results available for visualization.")


In [ ]:
# Plot: Loss curve + normalized target distribution on test set
import matplotlib.pyplot as plt
import numpy as np
import torch

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1) Loss vs Epochs
ax = axes[0]
if len(epoch_history) > 0 and len(train_loss_history) > 0:
    ax.plot(epoch_history, train_loss_history, color='tab:blue', linewidth=2, label='Train Loss')
    ax.set_title('Training Loss vs Epochs')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE, normalized)')
    ax.grid(True, alpha=0.3)
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No training history available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Training Loss vs Epochs')

# 2) Normalized target distribution on test set
ax = axes[1]
model.eval()
all_test_targets_scaled = []
all_test_preds_scaled = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device, non_blocking=True)
        out = model(
            batch.x,
            batch.edge_index,
            edge_attr=batch.edge_attr,
            batch=batch.batch,
            u=batch.u,
        )
        all_test_targets_scaled.append(batch.y_edge.detach().cpu().numpy().reshape(-1))
        all_test_preds_scaled.append(out.detach().cpu().numpy().reshape(-1))

if len(all_test_targets_scaled) > 0:
    target_values = np.concatenate(all_test_targets_scaled)
    pred_values = np.concatenate(all_test_preds_scaled)
    sample_idx = np.arange(target_values.shape[0])

    band_min = float(pred_values.min())
    band_max = float(pred_values.max())
    ax.axhspan(band_min, band_max, color='orange', alpha=0.16, zorder=0, label=f'Predicted range')

    ax.scatter(sample_idx, pred_values, s=10, alpha=0.35, color='tab:orange', label='Prediction (normalized)')
    ax.plot(sample_idx, np.sort(target_values), color='tab:blue', linewidth=2, label='Sorted target values')
    ax.set_title('Normalized Target Values on Test Set')
    ax.set_xlabel('Test samples (flattened edges)')
    ax.set_ylabel('Normalized target value')
    ax.grid(True, alpha=0.3)
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No test batches available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Normalized Target Values on Test Set')

plt.tight_layout()
plt.show()

print('✅ Training visualizations generated.')


# EXPORT

# EVALUATION & VISUALIZATION FOR OVER/UNDERFITTING

In [ ]:
"""Setup for visualization cells - prepare environment and imports"""
import matplotlib.pyplot as plt
import numpy as np
import torch
import config
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Environment ready for visualization.")


In [ ]:
# Predictions vs Actual + Residual diagnostics (Train/Test)
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

def _collect_preds_trues_original(loader):
    model.eval()
    pred_batches_local = []
    true_batches_local = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device, non_blocking=True)
            out = model(
                batch.x,
                batch.edge_index,
                edge_attr=batch.edge_attr,
                batch=batch.batch,
                u=batch.u,
            )
            pred_batches_local.append(out.detach().cpu().numpy())
            true_batches_local.append(batch.y_edge.detach().cpu().numpy())

    preds_scaled_local = np.concatenate(pred_batches_local, axis=0)
    trues_scaled_local = np.concatenate(true_batches_local, axis=0)

    preds_original_local = edge_target_scaler.inverse_transform(preds_scaled_local).reshape(-1)
    trues_original_local = edge_target_scaler.inverse_transform(trues_scaled_local).reshape(-1)
    return preds_original_local, trues_original_local

# Collect predictions in original kN scale
train_preds_original, train_trues_original = _collect_preds_trues_original(train_loader)
test_preds_original, test_trues_original = _collect_preds_trues_original(test_loader)

# Residuals and metrics
train_residuals = train_trues_original - train_preds_original
test_residuals = test_trues_original - test_preds_original

train_mae = float(mean_absolute_error(train_trues_original, train_preds_original))
test_mae = float(mean_absolute_error(test_trues_original, test_preds_original))
train_rmse = float(np.sqrt(mean_squared_error(train_trues_original, train_preds_original)))
test_rmse = float(np.sqrt(mean_squared_error(test_trues_original, test_preds_original)))
train_r2 = float(r2_score(train_trues_original, train_preds_original))
test_r2 = float(r2_score(test_trues_original, test_preds_original))

print(f"Train R2:  {train_r2:.4f}")
print(f"Test R2:   {test_r2:.4f}")
print(f"Train MAE: {train_mae:.4f} kN")
print(f"Test MAE:  {test_mae:.4f} kN")
print(f"Train RMSE: {train_rmse:.4f} kN")
print(f"Test RMSE:  {test_rmse:.4f} kN\n")

# 2x2 plot: Pred vs Actual + residuals for Train/Test
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

all_trues = np.concatenate([train_trues_original, test_trues_original])
all_preds = np.concatenate([train_preds_original, test_preds_original])
lim_low = min(all_trues.min(), all_preds.min())
lim_high = max(all_trues.max(), all_preds.max())

# Top-left: Train Pred vs Actual
ax = axes[0, 0]
ax.scatter(train_trues_original, train_preds_original, s=14, alpha=0.55, color="tab:blue", edgecolors="none", label="Train")
ax.plot([lim_low, lim_high], [lim_low, lim_high], "r--", linewidth=1.8, label="Perfect Prediction")
ax.set_title(f"Train Set: Predictions vs Actual\nR2 = {train_r2:.4f}")
ax.set_xlabel("Actual Force (kN)")
ax.set_ylabel("Predicted Force (kN)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

# Top-right: Test Pred vs Actual
ax = axes[0, 1]
ax.scatter(test_trues_original, test_preds_original, s=14, alpha=0.60, color="orange", edgecolors="none", label="Test")
ax.plot([lim_low, lim_high], [lim_low, lim_high], "r--", linewidth=1.8, label="Perfect Prediction")
ax.set_title(f"Test Set: Predictions vs Actual\nR2 = {test_r2:.4f}")
ax.set_xlabel("Actual Force (kN)")
ax.set_ylabel("Predicted Force (kN)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")

# Bottom-left: Train residuals vs predicted
ax = axes[1, 0]
ax.scatter(train_preds_original, train_residuals, s=14, alpha=0.55, color="tab:blue", edgecolors="none")
ax.axhline(0, color="red", linestyle="--", linewidth=1.8)
ax.set_title(f"Train Set: Residuals Plot\nMean Residual: {np.mean(train_residuals):.4f}")
ax.set_xlabel("Predicted Force (kN)")
ax.set_ylabel("Residuals (kN)")
ax.grid(True, alpha=0.3)

# Bottom-right: Test residuals vs predicted
ax = axes[1, 1]
ax.scatter(test_preds_original, test_residuals, s=14, alpha=0.60, color="orange", edgecolors="none")
ax.axhline(0, color="red", linestyle="--", linewidth=1.8)
ax.set_title(f"Test Set: Residuals Plot\nMean Residual: {np.mean(test_residuals):.4f}")
ax.set_xlabel("Predicted Force (kN)")
ax.set_ylabel("Residuals (kN)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
pred_residuals_fig = fig
plt.show()

print("✅ Prediction/residual diagnostic plots generated.")


In [ ]:
# Error distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Error distribution (train set)
ax = axes[0]
train_errors = np.abs(train_residuals)
ax.hist(train_errors, bins=50, alpha=0.7, edgecolor="black", color="blue", label="Train")
ax.axvline(train_mae, color="blue", linestyle="--", lw=2, label=f"Mean MAE: {train_mae:.4f}")
ax.set_xlabel("Absolute Error (kN)")
ax.set_ylabel("Frequency")
ax.set_title("Train Set: Error Distribution")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# 2. Error distribution (test set)
ax = axes[1]
test_errors = np.abs(test_residuals)
ax.hist(test_errors, bins=50, alpha=0.7, edgecolor="black", color="orange", label="Test")
ax.axvline(test_mae, color="orange", linestyle="--", lw=2, label=f"Mean MAE: {test_mae:.4f}")
ax.set_xlabel("Absolute Error (kN)")
ax.set_ylabel("Frequency")
ax.set_title("Test Set: Error Distribution")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
error_dist_fig = fig
plt.show()

print("✅ Error distribution plots generated.")


In [ ]:
# Final evaluation export
import importlib
from model_evaluation import save_evaluation

# Determine fit status
r2_gap = train_r2 - test_r2
if train_r2 < 0.7 and test_r2 < 0.7:
    status = "underfitting"
elif r2_gap > 0.05:
    status = "overfitting"
else:
    status = "good_fit"

metrics = {
    "train_r2": train_r2,
    "test_r2": test_r2,
    "train_mae": train_mae,
    "test_mae": test_mae,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "r2_gap": float(r2_gap),
}

# Model artifact details
model_artifact_stem = params["run_id"]
architecture_summary = {
    "model_class": "TrussEdgeNNConv",
    "node_in_dim": len(schema.node_continuous_cols) + len(schema.node_mask_cols),
    "edge_in_dim": len(schema.edge_feature_cols),
    "global_in_dim": len(schema.global_feature_cols),
    "hidden_dim": params["hidden_dim"],
    "edge_count": schema.edge_count,
    "device": str(device),
    "dataset_sources": {
        "node": params["node_csv"],
        "edge": params["edge_csv"],
        "global": params["global_csv"],
    },
}

experiment_notes = (
    f"USE_PRETRAINED={params['use_pretrained']}; "
    f"lr={params['learning_rate']}; epochs={params['epochs']}; "
    f"batch_size={params['batch_size']}; hidden_dim={params['hidden_dim']}; "
    f"weight_decay={params['weight_decay']}"
)

# Save evaluation
saved_files = save_evaluation(
    model_prefix=model_artifact_stem,
    dataset_name=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    metrics=metrics,
    pred_residuals_fig=pred_residuals_fig,
    error_dist_fig=error_dist_fig,
    training_visuals_fig=None,
    node_count=schema.node_count,
    edge_count=schema.edge_count,
    export_path=config.SM_DATA_PATH,
    status=status,
    run_id=params["run_id"],
    artifact_stem=model_artifact_stem,
    learning_rate=params["learning_rate"],
    epochs=params["epochs"],
    final_val_r2=final_val_r2,
    strict_dataset_label=f"{params['node_csv']} | {params['edge_csv']} | {params['global_csv']}",
    source_dataset_path=str(config.GH_DATA_PATH / params["edge_csv"]),
    architecture_summary=architecture_summary,
    experiment_notes=experiment_notes,
    train_split_ratio=params["train_split_ratio"],
    random_seed=params["random_seed"],
    source_notebook="c21_surrogate_model_training.ipynb",
)

print(f"\n✅ Evaluation exported:")
for f in saved_files:
    print(f"  - {f}")


## Interpretation Guide

### What to look for:

**OVERFITTING** 🔴 (Train performs much better than Test):
- Train R² >> Test R² (gap > 0.05)
- Train residuals are much smaller than test residuals
- Test error histogram has a heavier right tail
- Predictions vs Actual scatter: train points closer to red line than test points

**UNDERFITTING** 🔴 (Both train and test perform poorly):
- Both R² scores are low (< 0.7)
- Both residuals show large systematic patterns
- Both predictions scatter far from the red diagonal line
- High MAE/RMSE on both train and test

**GOOD FIT** ✅ (Train and Test perform similarly):
- Train and Test R² are close (gap < 0.05)
- Both residuals are centered around 0 with similar spread
- Both scatter plots show points close to diagonal line
- Error distributions are similar and centered

### Remedies:

If **Overfitting**:
- **Gather more training data** (most effective long-term solution—model memorizes less with more diverse samples)
- Add dropout layers to the model
- Increase weight decay (L2 regularization)
- Use early stopping on validation loss
- Try reducing model hidden_dim (e.g., 128 → 64)

If **Underfitting**:
- Increase hidden_dim (e.g., 128 → 256)
- Add more GNN layers
- Train for more epochs
- Check if the model has enough capacity for the problem

If **Good Fit**: ✅ Deploy and use in downstream tasks!